In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI

/Users/ankitdhandharia/Documents/Projects/RAG_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
llm = ChatOpenAI(
    model="liquid/lfm-2.5-1.2b-thinking:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

## **RAG IMPLEMENTATION WITH OUR OWN TEXT DATA**

### **STEP 1: Creating Embeddings For The Chunks**

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7348.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### **STEP 2: Retriving Embeddings from Vector Store**

In [7]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    embedding_function=embedding_model,
    persist_directory="./vector/"
)

/var/folders/dj/9vxldmmj3z1gbvb2bq1kqdvr0000gn/T/ipykernel_64051/2720840018.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


### **STEP 3: Semantic Search**

In [12]:
question = "What is contained in the tenth schedule of the constitution ?"
context = vectorstore.similarity_search(question, k=3)

In [13]:
context

[Document(metadata={'total_pages': 402, 'moddate': '2024-07-01T11:20:33+00:00', 'producer': 'iLovePDF', 'page': 375, 'source': './docs/content.pdf', 'page_label': '376', 'creationdate': '', 'creator': 'PyPDF'}, page_content='a nominated member of a House shall,—______________________________________________1. Tenth Schedule added by the Constitution (Fifty-second Amendment) Act, 1985, s. 6 (w.e.f. 1-3-1985).2. Certain words omitted by the Constitution (Ninety-first Amendment) Act, 2003, s. 5 (w.e.f. 1-1-2004).3. Subs. by s. 5, ibid., for "paragraphs 3, 4 and 5" (w.e.f. 1-1-2004).'),
 Document(metadata={'moddate': '2024-07-01T11:20:33+00:00', 'page_label': '142', 'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'source': './docs/content.pdf', 'total_pages': 402, 'page': 141}, page_content='111\nPART VII[The States in Part B of the First Schedule].        ______________________________________________Omittedby the Constitution (Seventh Amendment) Act, 1956, s. 29 and Sch. 

### **Talk to LLM**

In [14]:
response = llm.invoke(f"{question} You can answer using the following context: {context}")
print(response.content)

In [55]:
print(response)

content='The minimum age required to contest for Presidency, as mandated by the constitutional provisions referenced, is **35 years old**. This requirement is explicitly stated in the foundational eligibility criteria outlined in the provided context documents. \n\n\\boxed{35}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 566, 'prompt_tokens': 1245, 'total_tokens': 1811, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 595, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'liquid/lfm-2.5-1.2b-thinking-20260120:free', 'system_fingerprint': None, 'id': 'gen-1773937201-UWKgENJJ